# Phase 1: T1 / T2 features on the NBA 3.1 cohort calendar

Builds the 51 Phase 1 features (38 T1 activity + 13 T2 profile) for 294,542 NPIs x 28 claims cohorts (c1 to c28), windows aligned to each cohort anchor (`lookback_end_date`), then spot-checks 10 HCPs by hand. Plan: `01_veeva_link_plans.md`, Part 2.

**Refresh cadence:** Veeva Link is purchased data (HCP-level, slow-changing, like demographic data), not a continuous feed like claims. These notebooks are re-run only when a new Veeva delivery is bought and loaded, not monthly.

| Step | What it does | Original notebook | Check | When to run |
| --- | --- | --- | --- | --- |
| 1.1 | T1 / T2 features per cohort, manifest, schema lock | `71_veeva_link_phase1_features` | K1 to K7 | After Phase 0 passes |
| 1.2 | Spot check: 10 HCP profile cards vs Veeva Link web and `31_veeva_kol_list.csv`; exports the review template | `72_veeva_link_spot_check` | spot-check pass | Optional, after each rebuild |

Needs: Phase 0 outputs (spine, activity, config). See `02_veeva_link_features_qc.md` for the latest run and check results.

Note: step 1.2 reads `31_veeva_kol_list.csv` and writes `21_spot_check_review_template.csv` (path constants `KOL_CSV`, `WS_CSV` in its setup cell).

## Feature dictionary: what this notebook builds

Step 1.1 builds **51 features** (38 T1 activity + 13 T2 profile / coverage, including `veeva_coverage_status`) at the grain `NPI_NUMBER` x `COHORT_ID`, for 294,542 NPIs x 28 claims cohorts (c1 to c28) -> `features_phase1/t1_t2/COHORT_ID=<id>/`. Step 1.2 builds no features (it is a manual spot check).

**Common rules (all Veeva features)**
* **Source and filter:** latest rows of the frozen snapshot (`IS_LATEST_FLAG = 'Y'`, not deleted, `SYS_TENANT = 'US'` on person tables only). Data cut-off 2025-03-06 (hub load 2025-06-30).
* **HCP match:** `PROVIDER_ID` -> `source_provider_bhid_npi_mapping` (NPI) -> `SCIENTIFIC_EXPERT.NPI`. One row per NPI.
* **Coverage status:** `NOT_IN_VEEVA` / `IN_VEEVA_UNSCORED` / `SCORED` (`SCORED` = best in-scope decile 1 to 10 in `CLINICAL_LEADER`, `FOCUS_AREA` or `DIGITAL_REACH_FOCUS_AREA`).
* **Missing is not zero:** `NOT_IN_VEEVA` HCPs get null on every feature except the coverage flags; in-Veeva HCPs with no activity get 0. Deciles are never 0: unscored experts get null.
* **In scope:** focus names in `config/focus_dictionary_v1` (299 names). Text tags (trials `NAME`, publication `TITLE`, event `NAME` + `TALK_TITLE`) come from `config/keyword_list_v4` (209 patterns: 166 include, 43 exclude): `lower(text) LIKE pattern`; a tag is set when an include pattern matches and no exclude pattern of the same tag matches.
* **As-of windows:** window end `E = min(cohort anchor, 2025-03-06)`; trials and publications look back 5 years, events and payments 3 years. A payment program year `Y` counts only once published (30 June of `Y + 1`).
* **Transforms:** `log_n` = `log1p(count)`; flags are 0 / 1; composites = mean of the z-scores of their parts, standardised over the full HCP Universe of the cohort (not in Veeva = 0 for the statistics only; their composite stays null).
* **Wording and use:** say "influence / authority", not "KOL"; no payment feature in any field-facing output.

**Business logic in this phase**
* **Cohorts:** NBA 3.1 claims cohorts c1 to c28 (`cohorts/claims/cohorts.parquet`), anchor = `lookback_end_date` (1st of the month, c1 2024-05-01 to c28 2026-08-01).
* **Windows per cohort:** the windows end at min(anchor, 2025-03-06). Cohorts with the same window ends share one build (a "signature", 12 in total); c12 to c28 all end at 2025-03-06, so their values are identical (no Veeva data after the cut-off, F2).
* **Additions vs Phase 0:** per-indication counts (HF, CKD, T1D) for trials, publications and events. The specialty fallback column was dropped on 25 Nov 2026 (F3: 100% null, input I3 not delivered).
* **T3 is not joined here** (P1-D4); it lives in `30_`.
* **Schema lock:** columns are fixed in `features_phase1/schema_lock/columns.json`; checks K1 to K7.

**Features (51 columns)**

| # | Feature | Tier | Family | Source tables | Formula / rule | Window |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | `veeva_coverage_status` | key | coverage | spine + SCIENTIFIC_EXPERT + score tables | SCORED / IN_VEEVA_UNSCORED / NOT_IN_VEEVA; tells a true 0 from no Veeva data | - |
| 2 | `veeva_has_profile` | T2 | coverage | SCIENTIFIC_EXPERT | 1 if matched to SCIENTIFIC_EXPERT on NPI | - |
| 3 | `veeva_has_inscope_decile` | T2 | coverage | CLINICAL_LEADER, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA | 1 if any in-scope decile (dictionary v1) | - |
| 4 | `veeva_n_inscope_themes` | T2 | coverage | CLINICAL_LEADER, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA | distinct dictionary themes with a decile | - |
| 5 | `veeva_trial_any_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | any trial active in window | 5 years |
| 6 | `veeva_trial_log_n_5y` | T1 | trials | same | log1p distinct trials in window | 5 years |
| 7 | `veeva_trial_pi_flag_5y` | T1 | trials | same | PI on any trial in window | 5 years |
| 8 | `veeva_trial_n_inscope_5y` | T1 | trials | same | distinct in-scope trials in window | 5 years |
| 9 | `veeva_trial_n_inscope_hf_5y` | T1 | trials | same | distinct trials tagged HF in window | 5 years |
| 10 | `veeva_trial_n_inscope_ckd_5y` | T1 | trials | same | distinct trials tagged CKD in window | 5 years |
| 11 | `veeva_trial_n_inscope_t1d_5y` | T1 | trials | same | distinct trials tagged T1D in window | 5 years |
| 12 | `veeva_trial_max_phase_inscope_5y` | T1 | trials | same | max phase 1-4 of in-scope trials, 0 if none | 5 years |
| 13 | `veeva_trial_active_flag` | T1 | trials | same | trial running at window end | - |
| 14 | `veeva_trial_finerenone_flag_5y` | T1 | trials | same | trial name tagged finerenone | 5 years |
| 15 | `veeva_trial_mra_flag_5y` | T1 | trials | same | trial name tagged MRA or finerenone | 5 years |
| 16 | `veeva_trial_sglt2_flag_5y` | T1 | trials | same | trial name tagged SGLT2 | 5 years |
| 17 | `veeva_trial_cardiorenal_n_5y` | T1 | trials | same | distinct trials tagged CKD / cardiorenal / landmark / finerenone / MRA / SGLT2 / ERA | 5 years |
| 18 | `veeva_pub_any_5y` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | any publication in window | 5 years |
| 19 | `veeva_pub_log_n_5y` | T1 | publications | same | log1p publications in window | 5 years |
| 20 | `veeva_pub_senior_author_flag_5y` | T1 | publications | same | first or last author in window | 5 years |
| 21 | `veeva_pub_n_inscope_5y` | T1 | publications | same | in-scope publications in window | 5 years |
| 22 | `veeva_pub_n_inscope_hf_5y` | T1 | publications | same | publications tagged HF in window | 5 years |
| 23 | `veeva_pub_n_inscope_ckd_5y` | T1 | publications | same | publications tagged CKD in window | 5 years |
| 24 | `veeva_pub_n_inscope_t1d_5y` | T1 | publications | same | publications tagged T1D in window | 5 years |
| 25 | `veeva_pub_months_since_last` | T1 | publications | same | months from window end to last publication, null if none | - |
| 26 | `veeva_event_any_3y` | T1 | events | MEDICAL_EVENT, EVENT_ATTENDEE | any event in window | 3 years |
| 27 | `veeva_event_podium_flag_3y` | T1 | events | same | speaker / chair / panelist / moderator in window | 3 years |
| 28 | `veeva_event_poster_flag_3y` | T1 | events | same | poster presenter in window | 3 years |
| 29 | `veeva_event_n_inscope_3y` | T1 | events | same | distinct in-scope events in window | 3 years |
| 30 | `veeva_event_n_inscope_hf_3y` | T1 | events | same | distinct events tagged HF in window | 3 years |
| 31 | `veeva_event_n_inscope_ckd_3y` | T1 | events | same | distinct events tagged CKD in window | 3 years |
| 32 | `veeva_event_n_inscope_t1d_3y` | T1 | events | same | distinct events tagged T1D in window | 3 years |
| 33 | `veeva_pay_any_3y` | T1 | payments | PAYMENT | any non-Bayer payment, published program years in window | 3 years |
| 34 | `veeva_pay_log_total_3y` | T1 | payments | PAYMENT | log1p total USD, non-Bayer | 3 years |
| 35 | `veeva_pay_excl_food_log_3y` | T1 | payments | PAYMENT | log1p total USD excluding food and beverage, non-Bayer (null FOOD_FLAG = not food) | 3 years |
| 36 | `veeva_pay_n_companies_3y` | T1 | payments | PAYMENT | distinct paying companies, non-Bayer | 3 years |
| 37 | `veeva_pay_speaker_consult_flag_3y` | T1 | payments | PAYMENT | speaker / faculty / consulting payment, non-Bayer | 3 years |
| 38 | `veeva_pay_competitor_flag_3y` | T1 | payments | PAYMENT | payment from a competitor company (not Bayer) | 3 years |
| 39 | `veeva_pay_research_flag_3y` | T1 | payments | PAYMENT | research payment, non-Bayer | 3 years |
| 40 | `veeva_pay_bayer_flag_3y` | T1 | payments | PAYMENT | Bayer payment (separate flag only, D3) | 3 years |
| 41 | `veeva_expert_type` | T2 | profile | SCIENTIFIC_EXPERT | EXPERT_TYPE | - |
| 42 | `veeva_is_md_do` | T2 | profile | SCIENTIFIC_EXPERT | MD or DO in DEGREE_1..3 | - |
| 43 | `veeva_is_phd` | T2 | profile | SCIENTIFIC_EXPERT | PhD in DEGREE_1..3 | - |
| 44 | `veeva_is_academic` | T2 | profile | ADDRESS | affiliation looks academic | - |
| 45 | `veeva_emerging_expert` | T2 | profile | SCIENTIFIC_EXPERT | EMERGING_EXPERT = Y | - |
| 46 | `veeva_n_editorial_boards` | T2 | profile | ASSOCIATION | editorial board roles | - |
| 47 | `veeva_n_society_roles` | T2 | profile | ASSOCIATION | scientific / professional association roles | - |
| 48 | `veeva_kidney_society_flag` | T2 | profile | ASSOCIATION | association name mentions kidney / nephrology / renal | - |
| 49 | `veeva_n_sponsors` | T2 | profile | COMPANY_SPONSORSHIP | distinct sponsoring companies | - |
| 50 | `veeva_industry_engagement` | T1 | composite | rows 34, 36, 37 | mean z of pay_log_total, pay_n_companies, pay_speaker_consult (full HCP Universe of the cohort) | - |
| 51 | `veeva_network_breadth` | T1 | composite | rows 19, 26, 47 | mean z of pub_log_n, event_any, n_society_roles (full HCP Universe of the cohort) | - |

**Keys and metadata columns:** `NPI_NUMBER`, `PROVIDER_ID` (keys), `COHORT_ID` (partition), `cohort_set`, `veeva_window_end` (end of the activity windows), `veeva_data_age_months` (months from the data cut-off to the cohort anchor), `veeva_snapshot_id` (snapshot folder, `load=2025-06-30`).

**Outputs** under `OUT/features_phase1/`: `t1_t2/COHORT_ID=<id>/`, `feature_manifest/`, `cohort_windows/`, `checks/run=<ts>/`, `schema_lock/`.

# Phase 1: T1 / T2 features on the NBA 3.1 cohort calendar

Builds the 52 Phase 1 features (38 T1 activity + 14 T2 profile) for 294,542 NPIs x 43 cohorts, windows aligned to each cohort cut-off (offset 7), then spot-checks 10 HCPs by hand. Plan: `01_veeva_link_plans.md`, Part 2.

**Refresh cadence:** Veeva Link is purchased data (HCP-level, slow-changing, like demographic data), not a continuous feed like claims. These notebooks are re-run only when a new Veeva delivery is bought and loaded, not monthly.

| Step | What it does | Original notebook | Check | When to run |
| --- | --- | --- | --- | --- |
| 1.1 | T1 / T2 features per cohort, manifest, schema lock | `71_veeva_link_phase1_features` | K1 to K7 | After Phase 0 passes |
| 1.2 | Spot check: 10 HCP profile cards vs Veeva Link web and `31_veeva_kol_list.csv`; exports the review template | `72_veeva_link_spot_check` | spot-check pass | Optional, after each rebuild |

Needs: Phase 0 outputs (spine, activity, config). Last results: run `20261123T085209`, K1 to K7 all pass; spot check: 0 card values changed, KOL-list counts 0 differences.

Note: step 1.2 reads `31_veeva_kol_list.csv` and writes `72_veeva_link_review_template.csv` next to it (path constant `KOL_CSV` in its setup cell).

## Feature dictionary: what this notebook builds

Step 1.1 builds **52 features** (38 T1 activity + 14 T2 profile / coverage, including `veeva_coverage_status`) at the grain `NPI_NUMBER` x `COHORT_ID`, for 294,542 NPIs x 43 cohorts -> `features_phase1/t1_t2/COHORT_ID=<id>/`. Step 1.2 builds no features (it is a manual spot check).

**Common rules (all Veeva features)**
* **Source and filter:** latest rows of the frozen snapshot (`IS_LATEST_FLAG = 'Y'`, not deleted, `SYS_TENANT = 'US'` on person tables only). Data cut-off 2025-03-06 (hub load 2025-06-30).
* **HCP match:** `PROVIDER_ID` -> `source_provider_bhid_npi_mapping` (NPI) -> `SCIENTIFIC_EXPERT.NPI`. One row per NPI.
* **Coverage status:** `NOT_IN_VEEVA` / `IN_VEEVA_UNSCORED` / `SCORED` (`SCORED` = best in-scope decile 1 to 10 in `CLINICAL_LEADER`, `FOCUS_AREA` or `DIGITAL_REACH_FOCUS_AREA`).
* **Missing is not zero:** `NOT_IN_VEEVA` HCPs get null on every feature except the coverage flags; in-Veeva HCPs with no activity get 0. Deciles are never 0: unscored experts get null.
* **In scope:** focus names in `config/focus_dictionary_v1` (299 names). Text tags (trials `NAME`, publication `TITLE`, event `NAME` + `TALK_TITLE`) come from `config/keyword_list_v2`: `lower(text) LIKE pattern`; a tag is set when an include pattern matches and no exclude pattern of the same tag matches (v2 adds 13 MRA exclusions such as acne, PCOS, alopecia, PAH).
* **As-of windows:** window end `E = min(cut-off, table data cut-off)`; trials and publications look back 5 years, events and payments 3 years. A payment program year `Y` counts only once published (30 June of `Y + 1`).
* **Transforms:** `log_n` = `log1p(count)`; flags are 0 / 1; composites = mean of the z-scores of their parts, standardised over the matched NPIs.
* **Wording and use:** say "influence / authority", not "KOL"; no payment feature in any field-facing output.

**Business logic in this phase**
* **Cohorts:** NBA 3.1 C1 to C22 (cohort months 2025-01 to 2026-10) and NBA 3.0 C01 to C26 (2023-04 to 2025-05, used only for the E2 vs E3 leakage test). Offset 7: cut-off = month end + 6 days, `COHORT_ID = YYYY-MM_offset7`.
* **Windows per cohort:** the as-of windows above end at min(cohort cut-off, table data cut-off). Cohorts with the same window ends share one build (a "signature"), so most NBA 3.1 cohorts have identical values (Veeva data stop at 2025-03-06).
* **Additions vs Phase 0:** per-indication counts (HF, CKD, T1D) for trials, publications and events, and `veeva_specialty_fallback`.
* **T3 is not joined here** (P1-D4); it lives in `30_`.
* **Schema lock:** columns are fixed in `features_phase1/schema_lock/columns.json`; checks K1 to K7.

**Features (52 columns)**

| # | Feature | Tier | Family | Source tables | Formula / rule | Window | Status |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | `veeva_coverage_status` | key | coverage | spine + SCIENTIFIC_EXPERT + score tables (54_) | SCORED / IN_VEEVA_UNSCORED / NOT_IN_VEEVA; tells a true 0 from no Veeva data | - | in use |
| 2 | `veeva_has_profile` | T2 | coverage | SCIENTIFIC_EXPERT, CLINICAL_LEADER, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA | 1 if matched to SCIENTIFIC_EXPERT on NPI | - | kept |
| 3 | `veeva_has_inscope_decile` | T2 | coverage | SCIENTIFIC_EXPERT, CLINICAL_LEADER, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA | 1 if any in-scope decile (dictionary v1) | - | kept |
| 4 | `veeva_n_inscope_themes` | T2 | coverage | SCIENTIFIC_EXPERT, CLINICAL_LEADER, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA | distinct dictionary themes with a decile | - | kept |
| 5 | `veeva_trial_any_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | any trial active in window | 5 years | kept |
| 6 | `veeva_trial_log_n_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | log1p distinct trials in window | 5 years | kept |
| 7 | `veeva_trial_pi_flag_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | PI on any trial in window | 5 years | kept |
| 8 | `veeva_trial_n_inscope_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | distinct in-scope trials in window | 5 years | kept |
| 9 | `veeva_trial_n_inscope_hf_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | distinct trials tagged HF in window | 5 years | kept |
| 10 | `veeva_trial_n_inscope_ckd_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | distinct trials tagged CKD in window | 5 years | kept |
| 11 | `veeva_trial_n_inscope_t1d_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | distinct trials tagged T1D in window | 5 years | kept |
| 12 | `veeva_trial_max_phase_inscope_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | max phase 1-4 of in-scope trials, 0 if none | 5 years | kept |
| 13 | `veeva_trial_active_flag` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | trial running at window end | - | kept |
| 14 | `veeva_trial_finerenone_flag_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | trial name tagged finerenone | 5 years | kept |
| 15 | `veeva_trial_mra_flag_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | trial name tagged MRA or finerenone | 5 years | kept |
| 16 | `veeva_trial_sglt2_flag_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | trial name tagged SGLT2 | 5 years | kept |
| 17 | `veeva_trial_cardiorenal_n_5y` | T1 | trials | CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL | distinct trials tagged CKD / cardiorenal / landmark / finerenone / MRA / SGLT2 / ERA | 5 years | kept |
| 18 | `veeva_pub_any_5y` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | any publication in window | 5 years | kept |
| 19 | `veeva_pub_log_n_5y` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | log1p publications in window | 5 years | kept |
| 20 | `veeva_pub_senior_author_flag_5y` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | first or last author in window | 5 years | kept |
| 21 | `veeva_pub_n_inscope_5y` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | in-scope publications in window | 5 years | kept |
| 22 | `veeva_pub_n_inscope_hf_5y` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | publications tagged HF in window | 5 years | kept |
| 23 | `veeva_pub_n_inscope_ckd_5y` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | publications tagged CKD in window | 5 years | kept |
| 24 | `veeva_pub_n_inscope_t1d_5y` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | publications tagged T1D in window | 5 years | kept |
| 25 | `veeva_pub_months_since_last` | T1 | publications | PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION | months from window end to last publication, null if none | - | kept |
| 26 | `veeva_event_any_3y` | T1 | events | MEDICAL_EVENT, EVENT_ATTENDEE | any event in window | 3 years | kept |
| 27 | `veeva_event_podium_flag_3y` | T1 | events | MEDICAL_EVENT, EVENT_ATTENDEE | speaker / chair / panelist / moderator in window | 3 years | kept |
| 28 | `veeva_event_poster_flag_3y` | T1 | events | MEDICAL_EVENT, EVENT_ATTENDEE | poster presenter in window | 3 years | kept |
| 29 | `veeva_event_n_inscope_3y` | T1 | events | MEDICAL_EVENT, EVENT_ATTENDEE | distinct in-scope events in window | 3 years | kept |
| 30 | `veeva_event_n_inscope_hf_3y` | T1 | events | MEDICAL_EVENT, EVENT_ATTENDEE | distinct events tagged HF in window | 3 years | kept |
| 31 | `veeva_event_n_inscope_ckd_3y` | T1 | events | MEDICAL_EVENT, EVENT_ATTENDEE | distinct events tagged CKD in window | 3 years | kept |
| 32 | `veeva_event_n_inscope_t1d_3y` | T1 | events | MEDICAL_EVENT, EVENT_ATTENDEE | distinct events tagged T1D in window | 3 years | kept |
| 33 | `veeva_pay_any_3y` | T1 | payments | PAYMENT | any payment, published program years in window | 3 years | kept |
| 34 | `veeva_pay_log_total_3y` | T1 | payments | PAYMENT | log1p total USD | 3 years | kept |
| 35 | `veeva_pay_excl_food_log_3y` | T1 | payments | PAYMENT | log1p total USD excluding food and beverage | 3 years | kept |
| 36 | `veeva_pay_n_companies_3y` | T1 | payments | PAYMENT | distinct paying companies | 3 years | kept |
| 37 | `veeva_pay_speaker_consult_flag_3y` | T1 | payments | PAYMENT | speaker / faculty / consulting payment | 3 years | kept |
| 38 | `veeva_pay_competitor_flag_3y` | T1 | payments | PAYMENT | payment from a competitor company (not Bayer) | 3 years | kept |
| 39 | `veeva_pay_research_flag_3y` | T1 | payments | PAYMENT | research payment | 3 years | kept |
| 40 | `veeva_pay_bayer_flag_3y` | T1 | payments | PAYMENT | Bayer payment (separate flag only, D3) | 3 years | kept |
| 41 | `veeva_expert_type` | T2 | profile | SCIENTIFIC_EXPERT | EXPERT_TYPE | - | kept |
| 42 | `veeva_is_md_do` | T2 | profile | SCIENTIFIC_EXPERT | MD or DO in DEGREE_1..3 | - | kept |
| 43 | `veeva_is_phd` | T2 | profile | SCIENTIFIC_EXPERT | PhD in DEGREE_1..3 | - | kept |
| 44 | `veeva_is_academic` | T2 | profile | SCIENTIFIC_EXPERT | affiliation looks academic | - | kept |
| 45 | `veeva_emerging_expert` | T2 | profile | SCIENTIFIC_EXPERT | EMERGING_EXPERT = Y | - | kept |
| 46 | `veeva_n_editorial_boards` | T2 | profile | SCIENTIFIC_EXPERT | editorial board roles | - | kept |
| 47 | `veeva_n_society_roles` | T2 | profile | SCIENTIFIC_EXPERT | scientific / professional association roles | - | kept |
| 48 | `veeva_kidney_society_flag` | T2 | profile | SCIENTIFIC_EXPERT | association name mentions kidney / nephrology / renal | - | kept |
| 49 | `veeva_n_sponsors` | T2 | profile | SCIENTIFIC_EXPERT | distinct sponsoring companies | - | kept |
| 50 | `veeva_specialty_fallback` | T2 | profile | SCIENTIFIC_EXPERT | Veeva SPECIALTY_1 only where the NBA 3.1 specialty is missing (input I3) | - | waiting_I3 |
| 51 | `veeva_industry_engagement` | T1 | composite | other T1 features (PAYMENT, PUBLICATION, MEDICAL_EVENT, ASSOCIATION) | mean z of pay_log_total, pay_n_companies, pay_speaker_consult (matched NPIs of the cohort) | - | kept |
| 52 | `veeva_network_breadth` | T1 | composite | other T1 features (PAYMENT, PUBLICATION, MEDICAL_EVENT, ASSOCIATION) | mean z of pub_log_n, event_any, n_society_roles (matched NPIs of the cohort) | - | kept |

**Keys and metadata columns**

| # | Feature | Tier | Family | Source tables | Formula / rule | Window | Status |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | `NPI_NUMBER` | key | key | HCP Universe / spine (54_) | HCP NPI; one row per NPI per cohort | - | in use |
| 2 | `PROVIDER_ID` | key | key | HCP Universe / spine (54_) | NBA provider id of the NPI | - | in use |
| 3 | `cohort_set` | key | key | cohort calendar (71_) | Cohort set of the row (NBA 3.0 or NBA 3.1 cohorts) | - | in use |
| 4 | `veeva_window_end` | key | key | cohort_windows (71_) | End date of the activity windows used for this cohort (see features_phase1/cohort_windows) | - | in use |
| 5 | `veeva_data_age_months` | key | key | cohort_windows (71_) | Months between the Veeva data cut-off and the cohort cut-off | - | in use |
| 6 | `veeva_snapshot_id` | key | key | 52_ snapshot | Snapshot folder the features were built from (load=<hub load date>) | - | in use |

The `Status` column is the current decision from the feature dictionary (step 4.4, run `20261123T111314`).

## 1.1 T1 / T2 features on the cohort calendar

*Original notebook: `71_veeva_link_phase1_features`*

Implements tasks P1.1 to P4 of `70_veeva_link_phase1_working_plan.md`. Builds the Veeva T1 and T2 block at the NBA 3.1 grain **`NPI_NUMBER` x `COHORT_ID`** from the Phase 0 outputs (`52_` to `56_`).

* **Cohorts:** NBA 3.1 claims cohorts c1 to c28 from `cohorts/claims/cohorts.parquet` (Training c1 to c24, Testing c25 to c26, Holdout c27 to c28). Anchor = `lookback_end_date` (1st of the month, c1 2024-05-01 to c28 2026-08-01). The NBA 3.0 leakage-test cohorts are no longer built.
* **As-of rule:** T1 windows end at min(cohort anchor, table data cut-off 2025-03-06) and are 5 years for trials / publications and 3 years for events / payments. Payment program year `Y` counts once published (30 June of `Y + 1`).
* **Speed:** cohorts that share the same window ends share one build. Each distinct window set (a **signature**) is built once, staged, and copied to its cohorts.
* **Missing is not zero:** `NOT_IN_VEEVA` NPIs are null on every feature except the coverage flags.
* **T3 is not joined here** (P1-D4).

**Outputs** under `OUT/features_phase1/`: `t1_t2/COHORT_ID=<id>/`, `t1_static/` (E2), `feature_manifest/`, `cohort_windows/`, `checks/run=<ts>/`, `schema_lock/`.

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from dateutil.relativedelta import relativedelta
import pandas as pd, datetime as dt, calendar, json, os

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAPSHOT_ID = "load=2025-06-30"
SNAP, CFG, ACT = f"{OUT}snapshot/{SNAPSHOT_ID}/", f"{OUT}config/", f"{OUT}activity/"
P1 = f"{OUT}features_phase1/"
STAGE = f"{P1}_stage/"
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
rd = lambda t: spark.read.parquet(f"{SNAP}{t}")

# Optional main specialty source (input I3): a table or path with NPI_NUMBER and SPECIALTY. None = not available yet
SPECIALTY_SOURCE = None

spine = spark.read.parquet(f"{OUT}spine/veeva_npi_se_map")
CUTOFF = {r.table: dt.date.fromisoformat(r.data_cutoff) for r in spark.read.parquet(f"{SNAP}_manifest").collect()}
WIN = {r.families: r.months for r in spark.read.parquet(CFG + "windows_v1").collect()}
M_TRIAL_PUB, M_EVENT_PAY = WIN["trials; publications"], WIN["events; payments"]
CHILD_CUTOFF = min(CUTOFF[t] for t in ["clinical_trial", "publication", "medical_event", "payment"])
COMPETITORS = ["astrazeneca", "boehringer", "lilly", "janssen", "novo nordisk"]
print("table cut-offs:", {t: str(CUTOFF[t]) for t in ["clinical_trial", "publication", "medical_event", "payment", "scientific_expert"]},
      "| windows:", M_TRIAL_PUB, M_EVENT_PAY)

In [0]:
# NBA 3.1 claims cohorts (claims feature family, offset 0), same file as the claims HCP features use
COHORTS_DIR = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/cohorts/claims"
CLAIMS_COHORTS = spark.read.parquet(f"{COHORTS_DIR}/cohorts.parquet").toPandas()
COHORT_META = json.loads(dbutils.fs.head(f"{COHORTS_DIR}/metadata.json"))
assert COHORT_META["feature_family"] == "claims" and COHORT_META["offset_days"] == 0, COHORT_META
assert len(CLAIMS_COHORTS) == COHORT_META["cohort_count"] and (CLAIMS_COHORTS.offset_days == 0).all()
CLAIMS_COHORTS = CLAIMS_COHORTS.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)

def cohort_cutoff(lookback_end):
    """Anchor date of the cohort = claims lookback_end_date (inclusive end of every Veeva window)."""
    return pd.Timestamp(lookback_end).date()

# self-test: 12-month look-back ending one month before the prediction month (data lag 1); cut-offs increase
for r in CLAIMS_COHORTS.itertuples():
    pm, ls, le = (pd.Timestamp(x).date() for x in (r.prediction_month, r.lookback_start_date, r.lookback_end_date))
    assert le == pm - relativedelta(months=2) and ls == le - relativedelta(months=11), r.cohort_id
    assert cohort_cutoff(le) == le and le.day == 1, r.cohort_id
assert list(map(cohort_cutoff, CLAIMS_COHORTS.lookback_end_date)) == sorted(map(cohort_cutoff, CLAIMS_COHORTS.lookback_end_date))
assert str(cohort_cutoff(CLAIMS_COHORTS.lookback_end_date.iloc[0])) == "2024-05-01" and str(cohort_cutoff(CLAIMS_COHORTS.lookback_end_date.iloc[-1])) == "2026-08-01"

def windows_for(cut):
    """(start exclusive, end inclusive) per family, capped at the table cut-off; payment program years."""
    w = {}
    for fam, t, m in [("trials", "clinical_trial", M_TRIAL_PUB), ("publications", "publication", M_TRIAL_PUB), ("events", "medical_event", M_EVENT_PAY)]:
        end = min(cut, CUTOFF[t])
        w[fam] = ((pd.Timestamp(end) - pd.DateOffset(months=m)).date(), end)
    pend = min(cut, CUTOFF["payment"])
    last_y = pend.year - 1 if pend >= dt.date(pend.year, 6, 30) else pend.year - 2
    w["pay_years"] = tuple(range(last_y - M_EVENT_PAY // 12 + 1, last_y + 1))
    return w

rows = []
for r in CLAIMS_COHORTS.itertuples():
    cut = cohort_cutoff(r.lookback_end_date); w = windows_for(cut)
    rows.append({"COHORT_ID": r.cohort_id, "cohort_month": pd.Timestamp(r.prediction_month).date(), "cohort_cutoff": cut, "cohort_set": "nba_3_1",
                 "dataset": r.dataset, "lookback_start_date": pd.Timestamp(r.lookback_start_date).date(), "lookback_end_date": pd.Timestamp(r.lookback_end_date).date(),
                 "trials_start": w["trials"][0], "trials_end": w["trials"][1], "pubs_start": w["publications"][0], "pubs_end": w["publications"][1],
                 "events_start": w["events"][0], "events_end": w["events"][1], "pay_years": ",".join(map(str, w["pay_years"])),
                 "veeva_window_end": min(cut, CHILD_CUTOFF),
                 "veeva_data_age_months": max(0, (cut.year - CHILD_CUTOFF.year) * 12 + cut.month - CHILD_CUTOFF.month - (cut.day < CHILD_CUTOFF.day))})
cw = pd.DataFrame(rows)
sig_cols = ["trials_start", "trials_end", "pubs_start", "pubs_end", "events_start", "events_end", "pay_years"]
cw["sig"] = cw.groupby(sig_cols, sort=False).ngroup()
cw["pubs_left_cut"] = (cw.pubs_start < dt.date(2019, 1, 1)).astype(int)   # publication data starts 2019 (plan R3)
# reference cohorts: C_REF = first cohort whose windows are capped at the Veeva data cut-off (was 2025-02_offset7); C_LAST = list date (was 2026-10_offset7)
C_REF = cw[cw.veeva_window_end == CHILD_CUTOFF].COHORT_ID.iloc[0]
C_LAST = cw.COHORT_ID.iloc[-1]
print("C_REF:", C_REF, "| C_LAST:", C_LAST)
spark.createDataFrame(cw.astype({c: str for c in cw.columns if c not in ("sig", "veeva_data_age_months", "pubs_left_cut")})) \
     .write.mode("overwrite").parquet(P1 + "cohort_windows")
print("claims cohorts:", len(cw), "| datasets:", cw.dataset.value_counts().to_dict(), "| distinct window signatures:", cw.sig.nunique())
display(cw[["COHORT_ID", "dataset", "lookback_start_date", "lookback_end_date", "cohort_cutoff", "trials_end", "pubs_start", "events_end", "pay_years", "sig", "veeva_data_age_months", "pubs_left_cut"]])

In [0]:
dup = spine.filter("NPI IS NOT NULL").groupBy("NPI").agg(F.count("*").alias("n_providers"), F.collect_list("PROVIDER_ID").alias("providers"))
dup_log = dup.filter("n_providers > 1").toPandas()
# keep one row per NPI (lowest PROVIDER_ID); the same NPI always maps to the same SE_ID, so values are identical
npi_spine = (spine.filter("NPI IS NOT NULL")
             .withColumn("rk", F.row_number().over(__import__("pyspark.sql.window", fromlist=["Window"]).Window.partitionBy("NPI").orderBy("PROVIDER_ID")))
             .filter("rk = 1").drop("rk")
             .select(F.col("NPI").alias("NPI_NUMBER"), "PROVIDER_ID", "SE_ID", "IN_VEEVA", "BEST_DECILE", "VEEVA_COVERAGE_STATUS"))
npi_spine.write.mode("overwrite").parquet(STAGE + "npi_spine")
npi_spine = spark.read.parquet(STAGE + "npi_spine")
matched = npi_spine.filter("IN_VEEVA = 1").select("SE_ID").distinct()
N_NPI = npi_spine.count()
se_conflict = dup.filter("n_providers > 1").join(spine.select("NPI", "SE_ID"), "NPI").groupBy("NPI").agg(F.countDistinct("SE_ID").alias("n")).filter("n > 1").count()
print("providers dropped (no NPI):", spine.filter("NPI IS NULL").count(), "| NPIs on 2+ providers:", len(dup_log),
      "| of those with different SE_IDs:", se_conflict, "| NPI rows:", N_NPI, "| matched experts:", matched.count())
display(npi_spine.groupBy("VEEVA_COVERAGE_STATUS").count())

In [0]:
IND = ["hf", "ckd", "t1d"]
# F3 (25 Nov 2026): veeva_specialty_fallback dropped (100% null, input I3 never delivered); 51 features
FEATURE_SPEC = [
    ("veeva_coverage_status", "T2", "coverage", "NOT_IN_VEEVA / IN_VEEVA_UNSCORED / SCORED"),
    ("veeva_has_profile", "T2", "coverage", "1 if matched to SCIENTIFIC_EXPERT on NPI"),
    ("veeva_has_inscope_decile", "T2", "coverage", "1 if any in-scope decile (dictionary v1)"),
    ("veeva_n_inscope_themes", "T2", "coverage", "distinct dictionary themes with a decile"),
    ("veeva_trial_any_5y", "T1", "trials", "any trial active in window"),
    ("veeva_trial_log_n_5y", "T1", "trials", "log1p distinct trials in window"),
    ("veeva_trial_pi_flag_5y", "T1", "trials", "PI on any trial in window"),
    ("veeva_trial_n_inscope_5y", "T1", "trials", "distinct in-scope trials in window"),
    *[(f"veeva_trial_n_inscope_{i}_5y", "T1", "trials", f"distinct trials tagged {i.upper()} in window") for i in IND],
    ("veeva_trial_max_phase_inscope_5y", "T1", "trials", "max phase 1-4 of in-scope trials, 0 if none"),
    ("veeva_trial_active_flag", "T1", "trials", "trial running at window end"),
    ("veeva_trial_finerenone_flag_5y", "T1", "trials", "trial name tagged finerenone"),
    ("veeva_trial_mra_flag_5y", "T1", "trials", "trial name tagged MRA or finerenone"),
    ("veeva_trial_sglt2_flag_5y", "T1", "trials", "trial name tagged SGLT2"),
    ("veeva_trial_cardiorenal_n_5y", "T1", "trials", "distinct trials tagged CKD / cardiorenal / landmark / finerenone / MRA / SGLT2 / ERA"),
    ("veeva_pub_any_5y", "T1", "publications", "any publication in window"),
    ("veeva_pub_log_n_5y", "T1", "publications", "log1p publications in window"),
    ("veeva_pub_senior_author_flag_5y", "T1", "publications", "first or last author in window"),
    ("veeva_pub_n_inscope_5y", "T1", "publications", "in-scope publications in window"),
    *[(f"veeva_pub_n_inscope_{i}_5y", "T1", "publications", f"publications tagged {i.upper()} in window") for i in IND],
    ("veeva_pub_months_since_last", "T1", "publications", "months from window end to last publication, null if none"),
    ("veeva_event_any_3y", "T1", "events", "any event in window"),
    ("veeva_event_podium_flag_3y", "T1", "events", "speaker / chair / panelist / moderator in window"),
    ("veeva_event_poster_flag_3y", "T1", "events", "poster presenter in window"),
    ("veeva_event_n_inscope_3y", "T1", "events", "distinct in-scope events in window"),
    *[(f"veeva_event_n_inscope_{i}_3y", "T1", "events", f"distinct events tagged {i.upper()} in window") for i in IND],
    ("veeva_pay_any_3y", "T1", "payments", "any non-Bayer payment, published program years in window"),
    ("veeva_pay_log_total_3y", "T1", "payments", "log1p total USD, non-Bayer"),
    ("veeva_pay_excl_food_log_3y", "T1", "payments", "log1p total USD excluding food and beverage, non-Bayer (null FOOD_FLAG = not food)"),
    ("veeva_pay_n_companies_3y", "T1", "payments", "distinct paying companies, non-Bayer"),
    ("veeva_pay_speaker_consult_flag_3y", "T1", "payments", "speaker / faculty / consulting payment, non-Bayer"),
    ("veeva_pay_competitor_flag_3y", "T1", "payments", "payment from a competitor company (not Bayer)"),
    ("veeva_pay_research_flag_3y", "T1", "payments", "research payment, non-Bayer"),
    ("veeva_pay_bayer_flag_3y", "T1", "payments", "Bayer payment (separate flag only, D3)"),
    ("veeva_expert_type", "T2", "profile", "EXPERT_TYPE"),
    ("veeva_is_md_do", "T2", "profile", "MD or DO in DEGREE_1..3"),
    ("veeva_is_phd", "T2", "profile", "PhD in DEGREE_1..3"),
    ("veeva_is_academic", "T2", "profile", "affiliation looks academic"),
    ("veeva_emerging_expert", "T2", "profile", "EMERGING_EXPERT = Y"),
    ("veeva_n_editorial_boards", "T2", "profile", "editorial board roles"),
    ("veeva_n_society_roles", "T2", "profile", "scientific / professional association roles"),
    ("veeva_kidney_society_flag", "T2", "profile", "association name mentions kidney / nephrology / renal"),
    ("veeva_n_sponsors", "T2", "profile", "distinct sponsoring companies"),
    ("veeva_industry_engagement", "T1", "composite", "mean z of pay_log_total, pay_n_companies, pay_speaker_consult (full HCP Universe of the cohort, not in Veeva = 0 for the statistics)"),
    ("veeva_network_breadth", "T1", "composite", "mean z of pub_log_n, event_any, n_society_roles (full HCP Universe of the cohort, not in Veeva = 0 for the statistics)"),
]
COLS = [f[0] for f in FEATURE_SPEC]
T1_COLS = [f[0] for f in FEATURE_SPEC if f[1] == "T1"]
NON_ZERO_FILL = {"veeva_coverage_status", "veeva_has_profile", "veeva_has_inscope_decile", "veeva_expert_type",
                 "veeva_pub_months_since_last", "veeva_industry_engagement", "veeva_network_breadth"}
KEYS = ["NPI_NUMBER", "PROVIDER_ID"]
META = ["cohort_set", "veeva_window_end", "veeva_data_age_months", "veeva_snapshot_id"]
assert len(COLS) == len(set(COLS))
print("features:", len(COLS), "| T1:", len(T1_COLS), "| T2:", len(COLS) - len(T1_COLS))

In [0]:
fd = spark.read.parquet(CFG + "focus_dictionary_v1")
def scored(t, keys, dcol):
    k = fd.filter(f"source_table = '{t}'").select(F.col("focus_name").alias("FOCUS_NAME"), F.col("cl_type").alias("TYPE"), "theme")
    k = k if t == "clinical_leader" else k.drop("TYPE")
    return (rd(t).join(matched, "SE_ID").join(F.broadcast(k), keys)
              .select("SE_ID", "theme", F.col(dcol).cast("int").alias("D")).filter("D BETWEEN 1 AND 10"))
themes = (scored("clinical_leader", ["FOCUS_NAME", "TYPE"], "SE_DECILE").unionByName(scored("focus_area", ["FOCUS_NAME"], "FOCUS_DECILE"))
          .unionByName(scored("digital_reach_focus_area", ["FOCUS_NAME"], "SE_DECILE"))
          .groupBy("SE_ID").agg(F.countDistinct("theme").alias("veeva_n_inscope_themes")))

se = rd("scientific_expert").join(matched, "SE_ID")
deg = F.upper(F.concat_ws("|", "DEGREE_1", "DEGREE_2", "DEGREE_3"))
prof = se.select("SE_ID", F.col("EXPERT_TYPE").alias("veeva_expert_type"),
                 deg.rlike(r"(^|\|)(MD|DO)(\||$)").cast("int").alias("veeva_is_md_do"),
                 deg.rlike(r"(^|\|)PHD(\||$)").cast("int").alias("veeva_is_phd"),
                 (F.col("EMERGING_EXPERT") == "Y").cast("int").alias("veeva_emerging_expert"),
                 F.col("SPECIALTY_1").alias("VEEVA_SPECIALTY_1"))
acad = (rd("address").join(matched, "SE_ID").select("SE_ID", F.lower(F.concat_ws(" ", "AFFILIATION", "DEPARTMENT")).alias("aff"))
        .groupBy("SE_ID").agg(F.max(F.col("aff").rlike("universit|college|school of medicine|academic").cast("int")).alias("veeva_is_academic")))
assoc = (rd("association").join(matched, "SE_ID").groupBy("SE_ID").agg(
    F.sum((F.col("ASSOCIATION_TYPE") == "Editorial Board").cast("int")).alias("veeva_n_editorial_boards"),
    F.sum(F.col("ASSOCIATION_TYPE").isin("Scientific Association", "Professional Association").cast("int")).alias("veeva_n_society_roles"),
    F.max(F.lower(F.concat_ws(" ", "ASSOCIATION_NAME", "ASSOCIATION_ORIGINAL_NAME")).rlike("kidney|nephrolog|renal").cast("int")).alias("veeva_kidney_society_flag")))
spons = rd("company_sponsorship").join(matched, "SE_ID").groupBy("SE_ID").agg(F.countDistinct("COMPANY_NAME").alias("veeva_n_sponsors"))
static_se = prof.join(acad, "SE_ID", "left").join(assoc, "SE_ID", "left").join(spons, "SE_ID", "left").join(themes, "SE_ID", "left")
static_se.write.mode("overwrite").parquet(STAGE + "static_se")
static_se = spark.read.parquet(STAGE + "static_se")
print("profile rows:", static_se.count())   # F3: specialty fallback dropped, specialty comes from the NBA 3.1 claims features

In [0]:
trials, pubs, events = (spark.read.parquet(ACT + t) for t in ["trials", "publications", "events"])
CARDIO = ["TAG_CKD", "TAG_CARDIORENAL", "TAG_LANDMARK_TRIAL", "TAG_FINERENONE", "TAG_MRA", "TAG_SGLT2", "TAG_ERA"]

# payments pre-aggregated once to SE x program year x company (cuts 38M rows)
(spark.read.parquet(ACT + "payments").withColumn("AMOUNT", F.expr("try_cast(AMOUNT AS DOUBLE)"))
   .groupBy("SE_ID", F.col("PROGRAM_YEAR").cast("int").alias("PY"), "COMPANY_NAME")
   .agg(F.sum("AMOUNT").alias("AMT"), F.sum(F.when(F.coalesce(F.col("FOOD_FLAG"), F.lit(0)) == 0, F.col("AMOUNT")).otherwise(F.lit(0.0))).alias("AMT_NOFOOD"),  # F1 fix: null FOOD_FLAG = not food
        F.max("SPEAKER_CONSULT_FLAG").alias("SPK"), F.max("RESEARCH_FLAG").alias("RES"), F.max("BAYER_FLAG").alias("BAY"))
   .withColumn("COMP", (F.lower("COMPANY_NAME").rlike("|".join(COMPETITORS)) & (F.col("BAY") == 0)).cast("int"))
   .write.mode("overwrite").parquet(STAGE + "pay_sy"))
pay_sy = spark.read.parquet(STAGE + "pay_sy")

def t1_features(ts, te, ps, pe, es, ee, yrs):
    """T1 per SE_ID for one window set. Returns (DataFrame, look-ahead maxima)."""
    tw = trials.filter((F.col("START_DATE") <= F.lit(te)) & (F.col("END_DATE").isNull() | (F.col("END_DATE") >= F.lit(ts))))
    tr = tw.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_trial_any_5y"), F.log1p(F.countDistinct("CLINICAL_TRIAL_ID")).alias("veeva_trial_log_n_5y"),
        F.max("PI_FLAG").alias("veeva_trial_pi_flag_5y"),
        F.countDistinct(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("CLINICAL_TRIAL_ID"))).alias("veeva_trial_n_inscope_5y"),
        *[F.countDistinct(F.when(F.col(f"TAG_{i.upper()}") == 1, F.col("CLINICAL_TRIAL_ID"))).alias(f"veeva_trial_n_inscope_{i}_5y") for i in IND],
        F.coalesce(F.max(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("PHASE_NUM"))), F.lit(0)).alias("veeva_trial_max_phase_inscope_5y"),
        F.max((F.col("END_DATE").isNull() | (F.col("END_DATE") >= F.lit(te))).cast("int")).alias("veeva_trial_active_flag"),
        F.max("TAG_FINERENONE").alias("veeva_trial_finerenone_flag_5y"),
        F.max(F.greatest("TAG_MRA", "TAG_FINERENONE")).alias("veeva_trial_mra_flag_5y"),
        F.max("TAG_SGLT2").alias("veeva_trial_sglt2_flag_5y"),
        F.countDistinct(F.when(F.greatest(*CARDIO) == 1, F.col("CLINICAL_TRIAL_ID"))).alias("veeva_trial_cardiorenal_n_5y"))

    pw = pubs.filter((F.col("DATE") > F.lit(ps)) & (F.col("DATE") <= F.lit(pe)))
    pb = pw.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_pub_any_5y"), F.log1p(F.count("*")).alias("veeva_pub_log_n_5y"),
        F.max("SENIOR_AUTHOR_FLAG").alias("veeva_pub_senior_author_flag_5y"), F.sum("TAG_ANY_INSCOPE").alias("veeva_pub_n_inscope_5y"),
        *[F.sum(f"TAG_{i.upper()}").alias(f"veeva_pub_n_inscope_{i}_5y") for i in IND])
    last = (pubs.filter(F.col("DATE") <= F.lit(pe)).groupBy("SE_ID")
                .agg(F.floor(F.months_between(F.lit(pe), F.max("DATE"))).cast("int").alias("veeva_pub_months_since_last")))

    ew = events.filter((F.col("START_DATE") > F.lit(es)) & (F.col("START_DATE") <= F.lit(ee)))
    ev = ew.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_event_any_3y"), F.max("PODIUM_FLAG").alias("veeva_event_podium_flag_3y"),
        F.max("POSTER_FLAG").alias("veeva_event_poster_flag_3y"),
        F.countDistinct(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("MEDICAL_EVENT_ID"))).alias("veeva_event_n_inscope_3y"),
        *[F.countDistinct(F.when(F.col(f"TAG_{i.upper()}") == 1, F.col("MEDICAL_EVENT_ID"))).alias(f"veeva_event_n_inscope_{i}_3y") for i in IND])

    pw2 = pay_sy.filter(F.col("PY").isin(list(yrs)))
    # R10 (readiness principle 8, D3): Bayer payments stay in the personal block; every payment feature except the Bayer flag uses non-Bayer rows only
    nb = pw2.filter(F.coalesce(F.col("BAY"), F.lit(0)) == 0)
    bay = pw2.groupBy("SE_ID").agg(F.max("BAY").alias("veeva_pay_bayer_flag_3y"))
    py = nb.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_pay_any_3y"), F.log1p(F.sum("AMT")).alias("veeva_pay_log_total_3y"),
        F.log1p(F.sum("AMT_NOFOOD")).alias("veeva_pay_excl_food_log_3y"), F.countDistinct("COMPANY_NAME").alias("veeva_pay_n_companies_3y"),
        F.max("SPK").alias("veeva_pay_speaker_consult_flag_3y"), F.max("COMP").alias("veeva_pay_competitor_flag_3y"),
        F.max("RES").alias("veeva_pay_research_flag_3y"))

    look = {"max_trial_start": tw.agg(F.max("START_DATE")).first()[0], "max_pub_date": pw.agg(F.max("DATE")).first()[0],
            "max_event_start": ew.agg(F.max("START_DATE")).first()[0], "max_pay_year": pw2.agg(F.max("PY")).first()[0]}
    out = matched
    for p in [tr, pb, last, ev, py, bay]:
        out = out.join(p, "SE_ID", "left")
    return out, look

def zmean(df, cols, name):
    """R11 (readiness 5.1, U6): mean / SD over the full HCP Universe of the cohort (df = one row per NPI); HCPs not in Veeva count as 0
    for the statistics only, their composite stays null (principle 2)."""
    st = df.agg(*[F.mean(F.coalesce(F.col(c), F.lit(0))).alias(f"m_{c}") for c in cols],
                *[F.stddev(F.coalesce(F.col(c), F.lit(0))).alias(f"s_{c}") for c in cols]).collect()[0]
    return df.withColumn(name, sum((F.col(c) - st[f"m_{c}"]) / st[f"s_{c}"] for c in cols) / len(cols))

def assemble(t1_se):
    """NPI grain: coverage + T1 + T2 + composites, with the missing rule."""
    se_df = t1_se.join(static_se, "SE_ID", "left")
    h = (npi_spine.join(se_df, "SE_ID", "left")
            .withColumn("veeva_coverage_status", F.col("VEEVA_COVERAGE_STATUS"))
            .withColumn("veeva_has_profile", F.col("IN_VEEVA"))
            .withColumn("veeva_has_inscope_decile", F.col("BEST_DECILE").isNotNull().cast("int")))
    for c in [c for c in COLS if c not in NON_ZERO_FILL]:
        h = h.withColumn(c, F.when(F.col("IN_VEEVA") == 1, F.coalesce(F.col(c), F.lit(0))))
    h = zmean(h, ["veeva_pay_log_total_3y", "veeva_pay_n_companies_3y", "veeva_pay_speaker_consult_flag_3y"], "veeva_industry_engagement")
    h = zmean(h, ["veeva_pub_log_n_5y", "veeva_event_any_3y", "veeva_n_society_roles"], "veeva_network_breadth")
    return h.select(*KEYS, *COLS)

In [0]:
import time
look_rows = []
sigs = cw.drop_duplicates("sig").sort_values("sig")
for r in sigs.itertuples():
    t0 = time.time()
    t1_se, look = t1_features(r.trials_start, r.trials_end, r.pubs_start, r.pubs_end, r.events_start, r.events_end,
                              tuple(int(y) for y in r.pay_years.split(",")))
    assemble(t1_se).write.mode("overwrite").parquet(f"{STAGE}sig={r.sig}")
    look_rows.append({"sig": r.sig, **look, "seconds": round(time.time() - t0)})
look_df = pd.DataFrame(look_rows)
print("signatures built:", len(look_df), "| total minutes:", round(look_df.seconds.sum() / 60, 1))

# Static T1 for E2 (P1-D6): whole snapshot, no window; suffix _all
all_years = tuple(r[0] for r in pay_sy.select("PY").distinct().collect())
t1_all, look_all = t1_features(dt.date(1900, 1, 1), CHILD_CUTOFF, dt.date(1900, 1, 1), CUTOFF["publication"],
                               dt.date(1900, 1, 1), CUTOFF["medical_event"], all_years)
stat = assemble(t1_all).select(*KEYS, *[F.col(c).alias(c.replace("_5y", "_all").replace("_3y", "_all").replace("months_since_last", "months_since_last_all")) for c in T1_COLS])
stat.withColumn("veeva_snapshot_id", F.lit(SNAPSHOT_ID)).write.mode("overwrite").parquet(P1 + "t1_static")
print("t1_static rows:", spark.read.parquet(P1 + "t1_static").count(), "| payment program years used:", sorted(all_years))

In [0]:
from functools import reduce
stage_all = reduce(lambda a, b: a.unionByName(b), [spark.read.parquet(f"{STAGE}sig={s}").withColumn("sig", F.lit(int(s))) for s in sigs.sig])
nd = stage_all.agg(*[F.countDistinct(c).alias(c) for c in COLS]).toPandas().T[0]
constant = [c for c in COLS if nd[c] <= 1]
KEEP = [c for c in COLS if c not in constant]
print("constant features dropped:", constant or "none", "| kept:", len(KEEP))

cmap = spark.createDataFrame(cw[["COHORT_ID", "sig", "cohort_set", "veeva_window_end", "veeva_data_age_months"]].astype({"veeva_window_end": str}))
final = (stage_all.join(F.broadcast(cmap), "sig")
         .withColumn("veeva_window_end", F.to_date("veeva_window_end")).withColumn("veeva_snapshot_id", F.lit(SNAPSHOT_ID))
         .select(*KEYS, "COHORT_ID", *KEEP, *META))
final.write.mode("overwrite").partitionBy("COHORT_ID").parquet(P1 + "t1_t2")

spec = pd.DataFrame(FEATURE_SPEC, columns=["feature", "tier", "family", "rule"])
spec["indication"] = spec.feature.str.extract(r"_(hf|ckd|t1d)_")[0].fillna("kerendia_total")
spec["kept"] = spec.feature.isin(KEEP)
spec["status"] = ["dropped_constant" if f in constant else "kept" for f in spec.feature]
spec["E1"] = spec.family.eq("coverage"); spec["E2"] = spec.E1 | spec.tier.eq("T1"); spec["E3"] = spec.E2; spec["E4"] = True
spec["position"] = range(1, len(spec) + 1)
spec["snapshot_id"], spec["run_ts"] = SNAPSHOT_ID, RUN_TS
spark.createDataFrame(spec).write.mode("overwrite").parquet(P1 + "feature_manifest")
# count partitions through Spark (os.listdir on the volume can fail on serverless with 'Required key not available')
print("written cohorts:", spark.read.parquet(P1 + "t1_t2").select("COHORT_ID").distinct().count())

In [0]:
out = spark.read.parquet(P1 + "t1_t2")
exp_cols = KEYS + KEEP + META + ["COHORT_ID"]   # partition column comes last on read
checks = []
def add(k, name, value, expected, ok, note=""):
    checks.append({"check": k, "name": name, "value": str(value), "expected": str(expected), "pass": bool(ok), "note": note})

# K1 grain
g = out.groupBy("COHORT_ID").agg(F.count("*").alias("n"), F.countDistinct("NPI_NUMBER").alias("d")).toPandas()
add("K1", "one row per NPI per cohort", f"{g.n.min()}..{g.n.max()} rows, {len(g)} cohorts", f"{N_NPI} rows x {len(cw)} cohorts",
    len(g) == len(cw) and (g.n == N_NPI).all() and (g.d == N_NPI).all(),
    "Comparison with the NBA 3.1 claims NPI count not run: claims cohort output not readable yet")
# K2 fixed schema: manifest order now, and same as the previous run (schema_lock)
lock_p = P1 + "schema_lock/columns.json"
prev = json.load(open(lock_p)) if os.path.exists(lock_p) else None
add("K2", "columns = manifest order", len(out.columns), len(exp_cols), out.columns == exp_cols)
add("K2", "columns = previous run", "first run" if prev is None else (prev == out.columns), True if prev else "lock written",
    prev is None or prev == out.columns, "first run writes the lock" if prev is None else "")
os.makedirs(P1 + "schema_lock", exist_ok=True); json.dump(out.columns, open(lock_p, "w"))
# K3 no look-ahead
la = look_df.merge(sigs[["sig", "trials_end", "pubs_end", "events_end", "pay_years"]], on="sig")
bad = la[(pd.to_datetime(la.max_trial_start) > pd.to_datetime(la.trials_end)) | (pd.to_datetime(la.max_pub_date) > pd.to_datetime(la.pubs_end)) |
         (pd.to_datetime(la.max_event_start) > pd.to_datetime(la.events_end)) | (la.max_pay_year > la.pay_years.str.split(",").str[-1].astype(int))]
add("K3", "latest activity used <= window end", f"{len(bad)} signatures over", 0, len(bad) == 0)
# K4 missing is not zero
leak = out.filter("veeva_has_profile = 0").select(sum(F.col(c).isNotNull().cast("int") for c in KEEP if c not in NON_ZERO_FILL)).first()[0]
add("K4", "non-null T1/T2 values for NOT_IN_VEEVA", leak, 0, leak == 0)
# K5 cohort C_REF (first claims cohort capped at the data cut-off) vs 56_ anchor 2025-10-23 (both capped at 2025-03-06)
ref = spark.read.parquet(OUT + "features/t1_t2/anchor=2025-10-23")
common = [c for c in KEEP if c in ref.columns and c not in ("veeva_coverage_status", "veeva_expert_type")]
j = out.filter(F.col("COHORT_ID") == C_REF).select("PROVIDER_ID", *common).alias("a").join(ref.select("PROVIDER_ID", *common).alias("b"), "PROVIDER_ID")
tol = lambda c: 1e-3 if c in ("veeva_industry_engagement", "veeva_network_breadth") else 1e-9
mm = j.select(*[F.sum((~(F.abs(F.col(f"a.{c}") - F.col(f"b.{c}")) <= tol(c)) & ~(F.col(f"a.{c}").isNull() & F.col(f"b.{c}").isNull())).cast("int")).alias(c) for c in common]).toPandas().T[0]
add("K5", f"cohort {C_REF} = 56_ anchor 2025-10-23", f"{int(mm.sum())} mismatching values in {len(common)} columns, {j.count()} rows", 0,
    mm.sum() == 0, "; ".join(f"{c}: {int(v)}" for c, v in mm.items() if v > 0))
# K6 cohort identity: share of NPIs whose T1 values change vs the previous cohort
# composites are re-standardised per cohort, so they move for every matched NPI; keep them out of the identity hash
t1k = [c for c in KEEP if c in T1_COLS and c not in ("veeva_industry_engagement", "veeva_network_breadth")]
h = out.select("NPI_NUMBER", "COHORT_ID", F.xxhash64(*t1k).alias("h"))
cw_sorted = cw.sort_values("cohort_month").reset_index(drop=True)
pairs = spark.createDataFrame(pd.DataFrame({"COHORT_ID": cw_sorted.COHORT_ID[1:].values, "PREV": cw_sorted.COHORT_ID[:-1].values}))
chg = (h.join(pairs, "COHORT_ID").alias("x").join(h.alias("y"), (F.col("x.PREV") == F.col("y.COHORT_ID")) & (F.col("x.NPI_NUMBER") == F.col("y.NPI_NUMBER")))
        .groupBy(F.col("x.COHORT_ID").alias("COHORT_ID")).agg(F.avg((F.col("x.h") != F.col("y.h")).cast("double")).alias("share_changed")).toPandas())
chg = chg.merge(cw[["COHORT_ID", "cohort_set", "cohort_month"]], on="COHORT_ID").sort_values("cohort_month")
ref_month = cw.set_index("COHORT_ID").cohort_month[C_REF]
late31 = chg[chg.cohort_month > ref_month]
add("K6", f"T1 unchanged for claims cohorts after {C_REF}", f"max share changed {late31.share_changed.max():.4f}", 0, late31.share_changed.max() == 0,
    f"cohorts up to {C_REF} (cut-off before the data cut-off): mean share changed {chg[chg.cohort_month <= ref_month].share_changed.mean():.4f}")
# K7 constant features
add("K7", "constant features dropped", len(constant), "listed", True, ", ".join(constant))

ck = pd.DataFrame(checks).assign(run_ts=RUN_TS, snapshot_id=SNAPSHOT_ID)
spark.createDataFrame(ck).write.mode("overwrite").parquet(f"{P1}checks/run={RUN_TS}")
display(ck)
print("ALL CHECKS PASS:", bool(ck["pass"].all()))
display(chg)

In [0]:
num = [c for c in KEEP if c not in ("veeva_coverage_status", "veeva_expert_type")]
show = [cw.COHORT_ID.iloc[0], "c5", "c10", C_REF, C_LAST]
s = (out.filter(F.col("COHORT_ID").isin(show) & (F.col("veeva_has_profile") == 1)).groupBy("COHORT_ID")
        .agg(*[F.round(F.mean(c), 4).alias(c) for c in num]).toPandas().set_index("COHORT_ID").T)
display(s.reset_index().rename(columns={"index": "feature"}))
display(look_df)
if len(dup_log):
    display(dup_log)

## 1.2 Spot check

*Original notebook: `72_veeva_link_spot_check`*

Task P5 of `70_veeva_link_phase1_working_plan.md`: compare 10 known HCPs from `31_veeva_kol_list.csv` with their Veeva Link web profiles.

1. **Pick 10 HCPs**: one per profile type (rules in cell 3), all in the HCP Universe, chosen in a fixed order so a re-run picks the same people.
2. **Profile card per HCP**: identity and Veeva IDs, coverage and segment, the `71_` feature values (cohort `2026-10_offset7`, same as every NBA 3.1 cohort from 2025-02), and the underlying rows: trials, latest publications, latest events, and payments by year.
3. **Automatic cross-check**: the counts in `31_veeva_kol_list.csv` (an earlier extract) against the snapshot counts.
4. **Manual step**: open each HCP in Veeva Link (search by name or `LINK_ID`), compare with the card, and fill in `review_template` (`web_matches`, `mismatch_note`). Then copy the mismatches into section 8 of `70_`.

Output: `OUT/features_phase1/spot_check/` (`selection`, `cards`, `review_template`).

In [0]:
from pyspark.sql import functions as F, Window as W
import pandas as pd

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAP, ACT, P1 = f"{OUT}snapshot/load=2025-06-30/", f"{OUT}activity/", f"{OUT}features_phase1/"
SC = P1 + "spot_check/"
KOL_CSV = "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/Veeva Link To People Analysis/31_veeva_kol_list.csv"   # was wenjie.chen ..._sm (file no longer there)
COHORT = "c28"   # latest claims cohort (was 2026-10_offset7)
N_ROWS = 5   # rows shown per activity type on a card

kol = pd.read_csv(KOL_CSV, dtype={"SE_ID": str, "NPI": str})
feat = spark.read.parquet(P1 + "t1_t2").filter(F.col("COHORT_ID") == COHORT)
spine = spark.read.parquet(OUT + "spine/veeva_npi_se_map").select("NPI", "SE_ID", "BEST_DECILE", "CP_SEGMENT", "CP_SEG")
se = spark.read.parquet(SNAP + "scientific_expert").select("SE_ID", "LINK_ID", "VEEVA_ID", "FIRST_NAME", "LAST_NAME", "SPECIALTY_1", "DEGREE_1", "EXPERT_TYPE", "JOB_TITLE")

k = spark.createDataFrame(kol[["SE_ID", "NPI", "N_PUBLICATIONS", "N_TRIALS", "N_EVENTS"]]).withColumnRenamed("NPI", "NPI_NUMBER")
pool = (k.join(feat, "NPI_NUMBER").join(spine.withColumnRenamed("NPI", "NPI_NUMBER").drop("SE_ID"), "NPI_NUMBER", "left")
         .filter("veeva_has_profile = 1").toPandas())
print("KOL list rows:", len(kol), "| in HCP Universe and matched:", len(pool), "| SE_ID agrees with spine:",
      int(spark.createDataFrame(pool[["NPI_NUMBER", "SE_ID"]]).join(spine.withColumnRenamed("NPI", "NPI_NUMBER"), ["NPI_NUMBER", "SE_ID"]).count()))

In [0]:
# Each rule is tried in order; ties broken by NPI so the pick is repeatable; an HCP is used once
RULES = [
    ("MRA / finerenone trial investigator", lambda d: (d.veeva_trial_mra_flag_5y == 1) & (d.veeva_trial_pi_flag_5y == 1)),
    ("SGLT2 trial investigator",            lambda d: (d.veeva_trial_sglt2_flag_5y == 1)),
    ("Podium speaker, in-scope events",     lambda d: (d.veeva_event_podium_flag_3y == 1) & (d.veeva_event_n_inscope_3y >= 2)),
    ("Senior author, in-scope publications", lambda d: (d.veeva_pub_senior_author_flag_5y == 1) & (d.veeva_pub_n_inscope_5y >= 5)),
    ("Segment A, decile 10",                lambda d: (d.CP_SEG == "A") & (d.BEST_DECILE == 10)),
    ("Non-target, decile 9 to 10",          lambda d: (d.CP_SEG == "Non-target") & (d.BEST_DECILE >= 9)),
    ("Scored, no trials / pubs / events",   lambda d: (d.veeva_coverage_status == "SCORED") & (d.veeva_trial_any_5y == 0) & (d.veeva_pub_any_5y == 0) & (d.veeva_event_any_3y == 0)),
    ("Paid by a competitor, speaker / consult", lambda d: (d.veeva_pay_competitor_flag_3y == 1) & (d.veeva_pay_speaker_consult_flag_3y == 1)),
    ("Kidney society role",                 lambda d: d.veeva_kidney_society_flag == 1),
    ("Unscored, in Veeva",                  lambda d: d.veeva_coverage_status == "IN_VEEVA_UNSCORED"),
]
picked, used = [], set()
for name, rule in RULES:
    c = pool[rule(pool) & ~pool.NPI_NUMBER.isin(used)].sort_values("NPI_NUMBER")
    if len(c):
        r = c.iloc[0]; used.add(r.NPI_NUMBER)
        picked.append({"profile_type": name, "NPI_NUMBER": r.NPI_NUMBER, "SE_ID": r.SE_ID, "n_candidates": len(c)})
    else:
        picked.append({"profile_type": name, "NPI_NUMBER": None, "SE_ID": None, "n_candidates": 0})
sel = pd.DataFrame(picked)
sel_ok = sel.dropna(subset=["NPI_NUMBER"]).reset_index(drop=True)
sel_ok.insert(0, "card", range(1, len(sel_ok) + 1))
print("picked:", len(sel_ok), "of", len(RULES))
display(sel)

In [0]:
ids = spark.createDataFrame(sel_ok[["card", "profile_type", "NPI_NUMBER", "SE_ID"]])
se_ids = ids.select("SE_ID")

head = (ids.join(se, "SE_ID", "left").join(spine.withColumnRenamed("NPI", "NPI_NUMBER").drop("SE_ID"), "NPI_NUMBER", "left")
           .join(feat.drop("PROVIDER_ID"), "NPI_NUMBER", "left").orderBy("card").toPandas())

tags = ["TAG_FINERENONE", "TAG_MRA", "TAG_SGLT2", "TAG_CKD", "TAG_HF", "TAG_T2D", "TAG_T1D", "TAG_CARDIORENAL"]
def tag_list(df):
    return F.concat_ws(",", *[F.when(F.col(t) == 1, F.lit(t[4:].lower())) for t in tags])

def top(df, order, n=N_ROWS):
    w = W.partitionBy("SE_ID").orderBy(*order)
    return df.join(se_ids, "SE_ID").withColumn("rk", F.row_number().over(w)).filter(f"rk <= {n}").drop("rk")

tr = (top(spark.read.parquet(ACT + "trials"), [F.desc("TAG_ANY_INSCOPE"), F.desc("START_DATE")])
        .select("SE_ID", "NAME", "ROLE", "PHASE", "START_DATE", "END_DATE", tag_list(None).alias("tags")).toPandas())
pb = (top(spark.read.parquet(ACT + "publications"), [F.desc("DATE")])
        .select("SE_ID", "DATE", "AUTHORSHIP_POSITION", "JOURNAL", "TITLE", tag_list(None).alias("tags")).toPandas())
ev = (top(spark.read.parquet(ACT + "events"), [F.desc("START_DATE")])
        .select("SE_ID", "START_DATE", "POSITION", "EVENT_NAME", "TALK_TITLE", tag_list(None).alias("tags")).toPandas())
pay_all = spark.read.parquet(ACT + "payments").join(se_ids, "SE_ID").withColumn("AMOUNT", F.expr("try_cast(AMOUNT AS DOUBLE)"))
py = (pay_all.groupBy("SE_ID", "PROGRAM_YEAR").agg(F.count("*").alias("n_records"), F.round(F.sum("AMOUNT"), 0).alias("usd"),
                                                  F.countDistinct("COMPANY_NAME").alias("n_companies"), F.max("BAYER_FLAG").alias("bayer"))
        .orderBy("SE_ID", F.desc("PROGRAM_YEAR")).toPandas())
pc = (pay_all.groupBy("SE_ID", "COMPANY_NAME").agg(F.round(F.sum("AMOUNT"), 0).alias("usd"))
        .withColumn("rk", F.row_number().over(W.partitionBy("SE_ID").orderBy(F.desc("usd")))).filter("rk <= 3").drop("rk").toPandas())

# all-time snapshot counts for the cross-check with 31_veeva_kol_list.csv
cnt = (ids.select("SE_ID")
       .join(spark.read.parquet(ACT + "publications").groupBy("SE_ID").agg(F.count("*").alias("snap_pubs")), "SE_ID", "left")
       .join(spark.read.parquet(ACT + "trials").groupBy("SE_ID").agg(F.countDistinct("CLINICAL_TRIAL_ID").alias("snap_trials")), "SE_ID", "left")
       .join(spark.read.parquet(ACT + "events").groupBy("SE_ID").agg(F.countDistinct("MEDICAL_EVENT_ID").alias("snap_events")), "SE_ID", "left")
       .fillna(0).toPandas())
print("cards:", len(head), "| trial rows:", len(tr), "| publication rows:", len(pb), "| event rows:", len(ev), "| payment years:", len(py))

In [0]:
FEATS = ["veeva_coverage_status", "veeva_n_inscope_themes", "veeva_trial_any_5y", "veeva_trial_pi_flag_5y", "veeva_trial_n_inscope_5y",
         "veeva_trial_max_phase_inscope_5y", "veeva_trial_mra_flag_5y", "veeva_trial_sglt2_flag_5y", "veeva_pub_any_5y",
         "veeva_pub_senior_author_flag_5y", "veeva_pub_n_inscope_5y", "veeva_pub_months_since_last", "veeva_event_podium_flag_3y",
         "veeva_event_n_inscope_3y", "veeva_pay_any_3y", "veeva_pay_n_companies_3y", "veeva_pay_speaker_consult_flag_3y",
         "veeva_pay_competitor_flag_3y", "veeva_pay_bayer_flag_3y", "veeva_is_academic", "veeva_n_society_roles", "veeva_kidney_society_flag"]
for _, h in head.iterrows():
    s = h.SE_ID
    print("=" * 110)
    print(f"CARD {h.card}: {h.profile_type}")
    print(f"  {h.FIRST_NAME} {h.LAST_NAME} | NPI {h.NPI_NUMBER} | SE_ID {s} | LINK_ID {h.LINK_ID} | VEEVA_ID {h.VEEVA_ID}")
    print(f"  {h.EXPERT_TYPE} | {h.DEGREE_1} | {h.SPECIALTY_1} | {h.JOB_TITLE}")
    print(f"  segment {h.CP_SEGMENT} | best in-scope decile {h.BEST_DECILE} (T3, for review only)")
    print("  features (cohort " + COHORT + "): " + "; ".join(f"{f.replace('veeva_', '')}={h[f]}" for f in FEATS))
    for title, df in [("Trials (in-scope first)", tr), ("Latest publications", pb), ("Latest events", ev), ("Payments by program year", py), ("Top 3 paying companies", pc)]:
        d = df[df.SE_ID == s].drop(columns="SE_ID")
        print(f"  -- {title}: {len(d)} rows shown")
        if len(d):
            print(d.to_string(index=False, max_colwidth=70).replace("\n", "\n     ").join(["     ", ""]))

In [0]:
# 31_ has one row per SE_ID x source list (focus area / clinical leader / digital reach), so keep one row per SE_ID
kol1 = kol.drop_duplicates("SE_ID")[["SE_ID", "N_PUBLICATIONS", "N_TRIALS", "N_EVENTS"]]
x = (sel_ok.merge(kol1, on="SE_ID", how="left").merge(cnt, on="SE_ID", how="left"))
for a, b in [("N_PUBLICATIONS", "snap_pubs"), ("N_TRIALS", "snap_trials"), ("N_EVENTS", "snap_events")]:
    x[f"diff_{b[5:]}"] = x[b] - x[a]
print("Counts in 31_ come from an earlier extract; differences are expected to be small, not zero")
display(x[["card", "profile_type", "NPI_NUMBER", "N_PUBLICATIONS", "snap_pubs", "N_TRIALS", "snap_trials", "N_EVENTS", "snap_events",
           "diff_pubs", "diff_trials", "diff_events"]])

spark.createDataFrame(sel).write.mode("overwrite").parquet(SC + "selection")
cards = head[["card", "profile_type", "NPI_NUMBER", "SE_ID", "LINK_ID", "VEEVA_ID", "FIRST_NAME", "LAST_NAME", "SPECIALTY_1",
              "CP_SEGMENT", "BEST_DECILE"] + FEATS].merge(x[["SE_ID", "snap_pubs", "snap_trials", "snap_events"]], on="SE_ID")
spark.createDataFrame(cards.astype(str)).write.mode("overwrite").parquet(SC + "cards")
template = cards[["card", "profile_type", "NPI_NUMBER", "LINK_ID", "FIRST_NAME", "LAST_NAME"]].assign(
    web_matches="", mismatch_note="", reviewer="", review_date="")
spark.createDataFrame(template.astype(str)).write.mode("overwrite").parquet(SC + "review_template")
assert len(x) == len(sel_ok) == len(cards), (len(x), len(sel_ok), len(cards))
print("written:", SC, "| cards:", len(cards), "| SE_IDs repeated in 31_:", int(kol.SE_ID.duplicated().sum()))
display(template)

In [0]:
# One row per card: key features and the top activity row, for a quick side-by-side review
rows = []
for _, h in head.iterrows():
    s = h.SE_ID
    t, p, e, y, c = (df[df.SE_ID == s] for df in (tr, pb, ev, py, pc))
    rows.append({"card": h.card, "name": f"{h.FIRST_NAME} {h.LAST_NAME}", "degree_1": h.DEGREE_1, "specialty": h.SPECIALTY_1,
                 "segment": h.CP_SEGMENT, "decile": h.BEST_DECILE, "status": h.veeva_coverage_status, "themes": h.veeva_n_inscope_themes,
                 "trial_pi/inscope": f"{h.veeva_trial_pi_flag_5y}/{h.veeva_trial_n_inscope_5y}",
                 "pub_senior/inscope": f"{h.veeva_pub_senior_author_flag_5y}/{h.veeva_pub_n_inscope_5y}",
                 "podium/event_inscope": f"{h.veeva_event_podium_flag_3y}/{h.veeva_event_n_inscope_3y}",
                 "pay_cos/spk/comp/bayer": f"{h.veeva_pay_n_companies_3y}/{h.veeva_pay_speaker_consult_flag_3y}/{h.veeva_pay_competitor_flag_3y}/{h.veeva_pay_bayer_flag_3y}",
                 "top_trial": f"{t.iloc[0].NAME[:55]} ({t.iloc[0].ROLE}, {t.iloc[0].tags})" if len(t) else "",
                 "last_pub": f"{p.iloc[0].DATE} {p.iloc[0].TITLE[:50]}" if len(p) else "",
                 "last_event": f"{e.iloc[0].START_DATE} {e.iloc[0].POSITION}: {str(e.iloc[0].TALK_TITLE)[:40]}" if len(e) else "",
                 "payments_all_years": f"{int(y.PROGRAM_YEAR.min())}-{int(y.PROGRAM_YEAR.max())}, ${y.usd.sum():,.0f}" if len(y) else "",
                 "top_payer": f"{c.iloc[0].COMPANY_NAME} ${c.iloc[0].usd:,.0f}" if len(c) else ""})
display(pd.DataFrame(rows))

In [0]:
import os
# Review template as CSV: one copy next to the notebooks (open and fill in the workspace), one in the volume with the other outputs
REVIEW_COLS = ["web_matches", "mismatch_note", "reviewer", "review_date"]
WS_CSV = "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/Veeva Link To People Analysis/Implementation/21_spot_check_review_template.csv"   # Implementation copy; the original 72_ CSV is left untouched
VOL_CSV = SC.rstrip("/") + "/review_template.csv"

rt = spark.read.parquet(SC + "review_template").toPandas()
rt = rt.assign(card=rt.card.astype(int)).sort_values("card")   # card is stored as string; sort 1..10, not 1, 10, 2

# Do not overwrite a CSV that a reviewer has already started filling in
if os.path.exists(WS_CSV):
    old = pd.read_csv(WS_CSV, dtype=str).fillna("")
    if (old[REVIEW_COLS] != "").any().any():
        raise RuntimeError(f"{WS_CSV} already has review entries; rename or remove it before exporting again")

rt.to_csv(WS_CSV, index=False)
rt.to_csv(VOL_CSV, index=False)
chk = pd.read_csv(WS_CSV, dtype=str)
print(f"written: {WS_CSV}\n         {VOL_CSV}\nrows: {len(chk)} | columns: {list(chk.columns)}")
print("Fill in: web_matches (Y / N / partial), mismatch_note, reviewer, review_date (YYYY-MM-DD); look up each HCP in Veeva Link by LINK_ID")
display(chk)